# Classical path-filter runs (OFPF)

Nothing here builds a circuit. These measure where the path weight sits -- how far the
on-the-fly filter reaches, and how much of the sum the heaviest paths carry -- which is what
decides whether a truncated readout is worth anything on the circuit side. One environment
runs all of it; no cell needs a solver.

There is no Chain8 section: the chain is in the study to show that the construction runs at
d = 8, and its only measurement is a readout, which is a circuit-side run.

Each cell is one step of one system and writes its own files under `Data/<System>/<Property>/`,
exactly as `Running/OFPF/<System>.py` does from the command line. The equivalent command is printed above each
section. Cells carry no dependency on each other with one exception, noted where it applies, so
any single cell may be run on its own once **Master** has been run.

Every cell sets each module global it reads, including the ones another cell changes. A `.py`
driver never had to: it ran in a process of its own, so every module began at its default.


## Master

In [ ]:
import sys

master_dir   = '/home/bjbowling123/Project/CX_QuAPI'
sys.path[:0] = ['/home/bjbowling123/Project/PyQAKE/Code/Package', master_dir + '/Code']

# Seventeen measurement modules read sys.argv for an optional top N. A notebook's argv carries the
# kernel's connection file instead, so int(sys.argv[1]) would raise: it is emptied here and every
# module takes the default written in its own file.
sys.argv     = ['OFPF.ipynb']

from lib.cx_systems_lib import tls, sim2001, chain, ohmic
from measure.CX_coupling_sweep import LAT_TLS, LAT_3LS

# The published systems. TLS_STRONG is the weakly damped system's Hamiltonian on the bath that makes
# the rescaling bite (xi = 1.2, w_c = 2.5, beta = 0.2); CHAIN8 carries the weak system's bath,
# coupling and time step, so that d is the only difference between them.
TLS          = tls()
TLS_STRONG   = dict(tls(), name='TLS-strong', sd=ohmic(1.2, 2.5), beta=0.2)
SIM2001      = sim2001(0.72)
CHAIN8       = chain(8)

# Data/ and Output/ are resolved from lib/paths.py, not from the working directory, so a notebook
# writes exactly where the drivers write. CXQUAPI_ROOT in the environment moves both.
print(master_dir)

## TLS — weakly damped two-level system

`python3 OFPF/TLS.py`

In [ ]:
###===================================================<<< Settings

system       = 'TLS'
SY           = TLS

###===================================================<<< Calculation

# how many paths survive the on-the-fly filter, against N and the cutoff
from measure import CX_ofpf_reach
CX_ofpf_reach.SYSTEM = system
CX_ofpf_reach.CASES  = (SY,)
CX_ofpf_reach.main()

In [ ]:
###===================================================<<< Settings

system       = 'TLS'
SY           = TLS
N_top        = 8

###===================================================<<< Calculation

# how much of the sum the heaviest paths carry, for the populations
from measure import CX_path_dominance
CX_path_dominance.SYSTEM  = system
CX_path_dominance.CASES   = ((SY, N_top),)
CX_path_dominance.XI_SCAN = False      # the coupling scan spans systems; the Common section runs it
CX_path_dominance.main()

# and then for the coherences
from measure import CX_coherence_dominance
CX_coherence_dominance.SYSTEM  = system
CX_coherence_dominance.SYSTEMS = (SY,)
CX_coherence_dominance.main()

## TLS_strong — strongly damped two-level system

`python3 OFPF/TLS_strong.py`

In [ ]:
###===================================================<<< Settings

system       = 'TLS_strong'
SY           = TLS_STRONG

###===================================================<<< Calculation

# how many paths survive the on-the-fly filter, against N and the cutoff
from measure import CX_ofpf_reach
CX_ofpf_reach.SYSTEM = system
CX_ofpf_reach.CASES  = (SY,)
CX_ofpf_reach.main()

In [ ]:
###===================================================<<< Settings

system       = 'TLS_strong'
SY           = TLS_STRONG
N_top        = 8

###===================================================<<< Calculation

# how much of the sum the heaviest paths carry, for the populations
from measure import CX_path_dominance
CX_path_dominance.SYSTEM  = system
CX_path_dominance.CASES   = ((SY, N_top),)
CX_path_dominance.XI_SCAN = False      # the coupling scan spans systems; the Common section runs it
CX_path_dominance.main()

# and then for the coherences
from measure import CX_coherence_dominance
CX_coherence_dominance.SYSTEM  = system
CX_coherence_dominance.SYSTEMS = (SY,)
CX_coherence_dominance.main()

## Sim2001 — three-state model

`python3 OFPF/Sim2001.py`

In [ ]:
###===================================================<<< Settings

system       = 'Sim2001'
SY           = SIM2001

###===================================================<<< Calculation

# how many paths survive the on-the-fly filter, against N and the cutoff
from measure import CX_ofpf_reach
CX_ofpf_reach.SYSTEM = system
CX_ofpf_reach.CASES  = (SY,)
CX_ofpf_reach.main()

In [ ]:
###===================================================<<< Settings

system       = 'Sim2001'
SY           = SIM2001
N_top        = 6

###===================================================<<< Calculation

# how much of the sum the heaviest paths carry, for the populations
from measure import CX_path_dominance
CX_path_dominance.SYSTEM  = system
CX_path_dominance.CASES   = ((SY, N_top),)
CX_path_dominance.XI_SCAN = False      # the coupling scan spans systems; the Common section runs it
CX_path_dominance.main()

## Common — read across the systems

`python3 OFPF/Common.py`

These write to the systems the scan itself names — TLS, 3LS and TLS_shape — which is why no
single system's section owns them.

In [ ]:
###===================================================<<< Settings

system       = 'Common'

###===================================================<<< Calculation

# the coupling scan only: no per-system table, the sections above wrote those
from measure import CX_path_dominance
CX_path_dominance.SYSTEM  = system
CX_path_dominance.CASES   = ()
CX_path_dominance.XI_SCAN = True
CX_path_dominance.main()

In [ ]:
###===================================================<<< Settings

system       = 'Common'
N_shape      = 5          # the propagation length the bath-shape scan is read at

###===================================================<<< Calculation

# concentration against xi, then against the cutoff that sets the bath correlation time
from measure import CX_coupling_sweep, CX_markovian_shape
CX_coupling_sweep.main()
CX_markovian_shape.SYSTEM = system
CX_markovian_shape.main(N=N_shape)

In [ ]:
###===================================================<<< Settings

system       = 'Common'

###===================================================<<< Calculation

# lambda_min(M) and the largest full-window generator over the whole (beta, w_c dt) grid
from measure import CX_markovian_full_window
CX_markovian_full_window.main()